<a href="https://colab.research.google.com/github/pandeychetanya/HalloweenFlutter/blob/main/PD_pursuit_variability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trial-to-trial smooth-pursuit variability in Parkinson's disease

**Retained analyses**
- primary binocular 500–700 ms pursuit-gain reconstruction and cleaning
- participant-level IQR, MAD and SD
- Mann–Whitney test, Cliff's delta, exact label permutation and participant bootstrap
- leave-one-participant-out analysis
- adjustment for typical pursuit gain
- validation against Wu et al. supplementary pursuit velocity
- 12-specification cleaning sensitivity analysis
- within-participant trial-resampling uncertainty
- trial-order dynamics
- final main and supplementary figures
- graphical abstract

**Removed only because redundant or abandoned**
- repeated imports and path setup
- duplicate helper definitions
- early raw exploratory QC that was superseded by the final cleaning pipeline
- abandoned left-versus-right directional-asymmetry exploration
- duplicate versions of the same publication figures

The statistical definitions, thresholds and manuscript-facing analysis are unchanged.

In [2]:
# 1. SETUP

from pathlib import Path
from itertools import combinations
import platform
import warnings

import numpy as np
import pandas as pd
import scipy
import scipy.io as sio
from scipy.stats import mannwhitneyu, pearsonr, spearmanr, ttest_ind

import statsmodels
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

warnings.filterwarnings("ignore")

from google.colab import drive
drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/Wu_PD")
OUT = BASE / "analysis_output"
FIGDIR = OUT / "journal_figures"

OUT.mkdir(parents=True, exist_ok=True)
FIGDIR.mkdir(parents=True, exist_ok=True)

MAT_FILES = sorted(
    f for f in BASE.rglob("*.mat")
    if f.stem.upper().startswith(("PD", "NC"))
)

TARGET_SPEED = 10.0
WINDOW_START = 0.500
WINDOW_END = 0.700

PRIMARY_ABS_LIMIT = 30.0
PRIMARY_ROBUST_K = 6.0
MIN_CLEAN_SAMPLES = 5

print("PD + NC .mat files found:", len(MAT_FILES))
if len(MAT_FILES) != 20:
    print("WARNING: expected 20 files (10 PD + 10 NC).")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PD + NC .mat files found: 20


## 2. Shared raw-data functions

The released MATLAB fields are handled once here and reused throughout the notebook. The position-like fields are used only to select the monotonic trial time axis. The analysis window is always 500–700 ms after pursuit onset.

In [3]:
# 2. SHARED FUNCTIONS

def get_field(trial, field):
    if not hasattr(trial, field):
        return None
    try:
        return np.asarray(getattr(trial, field)).squeeze()
    except Exception:
        return None


def as_numeric(x):
    if x is None:
        return None
    try:
        x = np.asarray(x, dtype=float).squeeze()
        if x.ndim == 0:
            return np.array([float(x)])
        return x.ravel()
    except Exception:
        return None


def scalar_numeric(x):
    if x is None:
        return np.nan
    try:
        return float(np.asarray(x).squeeze())
    except Exception:
        return np.nan


def time_score(arr, pursuit_time):
    if arr is None or len(arr) < 10:
        return -np.inf

    x = np.asarray(arr, dtype=float)
    x = x[np.isfinite(x)]

    if len(x) < 10:
        return -np.inf

    dx = np.diff(x)
    if len(dx) == 0:
        return -np.inf

    increasing = np.mean(dx > 0)
    score = 5 * increasing

    if increasing > 0.95:
        score += 5

    if np.isfinite(pursuit_time):
        xmin, xmax = np.nanmin(x), np.nanmax(x)

        if xmin <= pursuit_time <= xmax:
            score += 4
        elif xmin <= pursuit_time / 1000 <= xmax:
            score += 2
        elif xmin / 1000 <= pursuit_time <= xmax / 1000:
            score += 2

    return score


def choose_time_vector(trial, pursuit_time):
    candidates = {
        "leftEyexXPos": as_numeric(get_field(trial, "leftEyexXPos")),
        "rightEyeXPos": as_numeric(get_field(trial, "rightEyeXPos")),
    }

    scores = {
        name: time_score(arr, pursuit_time)
        for name, arr in candidates.items()
    }

    best = max(scores, key=scores.get)

    if not np.isfinite(scores[best]):
        return None, None

    return candidates[best], best


def convert_time(time, pursuit_time):
    time = np.asarray(time, dtype=float)

    dt = np.diff(time)
    good_dt = dt[np.isfinite(dt) & (dt > 0)]

    if len(good_dt) == 0:
        return None, np.nan, np.nan

    median_dt = np.nanmedian(good_dt)

    if median_dt > 0.5:
        time_sec = time / 1000.0
        pursuit_sec = pursuit_time / 1000.0
    else:
        time_sec = time.copy()
        pursuit_sec = pursuit_time

        if (
            np.isfinite(pursuit_sec)
            and pursuit_sec > np.nanmax(time_sec) * 5
        ):
            pursuit_sec /= 1000.0

    dt_sec = np.diff(time_sec)
    dt_sec = dt_sec[np.isfinite(dt_sec) & (dt_sec > 0)]

    sampling_hz = (
        1 / np.nanmedian(dt_sec)
        if len(dt_sec)
        else np.nan
    )

    return time_sec, pursuit_sec, sampling_hz


def mad_unscaled(x):
    x = np.asarray(x, dtype=float)
    med = np.nanmedian(x)
    return np.nanmedian(np.abs(x - med))


def iqr_value(x):
    x = np.asarray(x, dtype=float)
    return np.nanpercentile(x, 75) - np.nanpercentile(x, 25)


def cliffs_delta(pd_values, nc_values):
    pd_values = np.asarray(pd_values, dtype=float)
    nc_values = np.asarray(nc_values, dtype=float)

    greater = sum(x > y for x in pd_values for y in nc_values)
    lower = sum(x < y for x in pd_values for y in nc_values)

    return (greater - lower) / (len(pd_values) * len(nc_values))

## 3. Extract raw binocular pursuit windows

The 500–700 ms binocular velocity samples are extracted once from the MATLAB files. Every primary and sensitivity cleaning specification is applied to these same raw windows.

In [4]:
# 3. EXTRACT RAW 500–700 ms BINOCULAR WINDOWS

raw_trials = []

for f in MAT_FILES:
    participant_id = f.stem.split("_")[0]
    group = "PD" if participant_id.upper().startswith("PD") else "NC"

    mat = sio.loadmat(
        f,
        squeeze_me=True,
        struct_as_record=False,
    )

    if "BinocularTrials" not in mat:
        continue

    trials = np.atleast_1d(mat["BinocularTrials"])

    for trial_number, trial in enumerate(trials, start=1):
        pursuit_time = scalar_numeric(
            get_field(trial, "timeToPursuit")
        )

        left_vel = as_numeric(get_field(trial, "leftEyeVel"))
        right_vel = as_numeric(get_field(trial, "rightEyeVel"))

        time_raw, time_field = choose_time_vector(
            trial,
            pursuit_time,
        )

        record = {
            "Participant": participant_id,
            "Group": group,
            "Trial": trial_number,
            "Time_field": time_field,
            "Sampling_Hz": np.nan,
            "N_window": 0,
            "Left_window": np.array([], dtype=float),
            "Right_window": np.array([], dtype=float),
        }

        if time_raw is None or not np.isfinite(pursuit_time):
            raw_trials.append(record)
            continue

        time_sec, pursuit_sec, sampling_hz = convert_time(
            time_raw,
            pursuit_time,
        )

        if time_sec is None:
            raw_trials.append(record)
            continue

        record["Sampling_Hz"] = sampling_hz

        mask = (
            np.isfinite(time_sec)
            & (time_sec >= pursuit_sec + WINDOW_START)
            & (time_sec <= pursuit_sec + WINDOW_END)
        )

        record["N_window"] = int(mask.sum())

        if left_vel is not None:
            n = min(len(time_sec), len(left_vel))
            valid = mask[:n] & np.isfinite(left_vel[:n])
            record["Left_window"] = np.asarray(
                left_vel[:n][valid],
                dtype=float,
            )

        if right_vel is not None:
            n = min(len(time_sec), len(right_vel))
            valid = mask[:n] & np.isfinite(right_vel[:n])
            record["Right_window"] = np.asarray(
                right_vel[:n][valid],
                dtype=float,
            )

        raw_trials.append(record)

print("Binocular trials extracted:", len(raw_trials))
print("Expected:", 200)

Binocular trials extracted: 200
Expected: 200


## 4. Primary cleaning and trial-level pursuit gain

The primary rule removes only high-speed contamination: speed >30°/s **or** speed > median + 6 robust SD, with robust SD = 1.4826 × MAD. Low velocities are retained. At least five cleaned samples are required per contributing eye.

This is a pragmatic approximation to Wu et al.'s saccade/blink removal, not an exact recreation of their event pipeline.

In [5]:
# 4. CLEANING FUNCTIONS + PRIMARY CLEANED TRIAL DATA

def clean_eye_velocity(values, abs_limit, robust_k):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    speed = np.abs(values)

    n_raw = len(speed)

    if n_raw == 0:
        return {
            "raw_mean": np.nan,
            "raw_median": np.nan,
            "clean_mean": np.nan,
            "clean_median": np.nan,
            "n_raw": 0,
            "n_clean": 0,
            "fraction_removed": np.nan,
            "max_speed": np.nan,
        }

    raw_mean = np.nanmean(speed)
    raw_median = np.nanmedian(speed)
    max_speed = np.nanmax(speed)

    mad = np.nanmedian(np.abs(speed - raw_median))
    robust_sigma = 1.4826 * mad

    robust_limit = (
        raw_median + robust_k * robust_sigma
        if np.isfinite(robust_sigma) and robust_sigma > 0
        else np.inf
    )

    contaminated = (
        (speed > abs_limit)
        | (speed > robust_limit)
    )

    clean = speed[~contaminated]
    n_clean = len(clean)

    fraction_removed = (n_raw - n_clean) / n_raw

    if n_clean >= MIN_CLEAN_SAMPLES:
        clean_mean = np.nanmean(clean)
        clean_median = np.nanmedian(clean)
    else:
        clean_mean = np.nan
        clean_median = np.nan

    return {
        "raw_mean": raw_mean,
        "raw_median": raw_median,
        "clean_mean": clean_mean,
        "clean_median": clean_median,
        "n_raw": n_raw,
        "n_clean": n_clean,
        "fraction_removed": fraction_removed,
        "max_speed": max_speed,
    }


def process_specification(raw_trials, abs_limit, robust_k):
    rows = []

    for rec in raw_trials:
        left = clean_eye_velocity(
            rec["Left_window"],
            abs_limit,
            robust_k,
        )
        right = clean_eye_velocity(
            rec["Right_window"],
            abs_limit,
            robust_k,
        )

        raw_means = [
            x["raw_mean"] for x in (left, right)
            if np.isfinite(x["raw_mean"])
        ]
        raw_medians = [
            x["raw_median"] for x in (left, right)
            if np.isfinite(x["raw_median"])
        ]
        clean_means = [
            x["clean_mean"] for x in (left, right)
            if np.isfinite(x["clean_mean"])
        ]
        clean_medians = [
            x["clean_median"] for x in (left, right)
            if np.isfinite(x["clean_median"])
        ]
        fractions = [
            x["fraction_removed"] for x in (left, right)
            if np.isfinite(x["fraction_removed"])
        ]

        raw_mean_gain = (
            np.mean(raw_means) / TARGET_SPEED
            if raw_means else np.nan
        )
        raw_median_gain = (
            np.mean(raw_medians) / TARGET_SPEED
            if raw_medians else np.nan
        )
        clean_mean_gain = (
            np.mean(clean_means) / TARGET_SPEED
            if clean_means else np.nan
        )
        clean_median_gain = (
            np.mean(clean_medians) / TARGET_SPEED
            if clean_medians else np.nan
        )

        max_candidates = np.asarray(
            [left["max_speed"], right["max_speed"]],
            dtype=float,
        )
        max_speed = (
            np.nanmax(max_candidates)
            if np.isfinite(max_candidates).any()
            else np.nan
        )

        rows.append({
            "Participant": rec["Participant"],
            "Group": rec["Group"],
            "Trial": rec["Trial"],
            "Time_field": rec["Time_field"],
            "Sampling_Hz": rec["Sampling_Hz"],
            "N_window": rec["N_window"],

            "Left_raw_mean": left["raw_mean"],
            "Left_raw_median": left["raw_median"],
            "Left_clean_mean": left["clean_mean"],
            "Left_clean_median": left["clean_median"],
            "Left_n_raw": left["n_raw"],
            "Left_n_clean": left["n_clean"],
            "Left_frac_removed": left["fraction_removed"],
            "Left_max_speed": left["max_speed"],

            "Right_raw_mean": right["raw_mean"],
            "Right_raw_median": right["raw_median"],
            "Right_clean_mean": right["clean_mean"],
            "Right_clean_median": right["clean_median"],
            "Right_n_raw": right["n_raw"],
            "Right_n_clean": right["n_clean"],
            "Right_frac_removed": right["fraction_removed"],
            "Right_max_speed": right["max_speed"],

            "Raw_mean_gain": raw_mean_gain,
            "Raw_median_gain": raw_median_gain,
            "Clean_mean_gain": clean_mean_gain,
            "Clean_median_gain": clean_median_gain,
            "Mean_fraction_removed": (
                np.mean(fractions) if fractions else np.nan
            ),
            "Max_speed": max_speed,
            "Clean_valid": np.isfinite(clean_mean_gain),
        })

    return pd.DataFrame(rows)


trial_df = process_specification(
    raw_trials,
    PRIMARY_ABS_LIMIT,
    PRIMARY_ROBUST_K,
)

valid_trials = trial_df[
    trial_df["Clean_mean_gain"].notna()
].copy()

print("Clean valid:", len(valid_trials), "/", len(trial_df))

print("\nFraction removed by group:")
print(
    valid_trials.groupby("Group")["Mean_fraction_removed"]
    .agg(["mean", "median", "max"])
)

trial_file = OUT / "Wu_binocular_pursuit_cleaning_QC.csv"
trial_df.to_csv(trial_file, index=False)

participant_qc = (
    valid_trials
    .groupby(["Participant", "Group"])
    .agg(
        N_trials=("Clean_mean_gain", "count"),
        Mean_clean_gain=("Clean_mean_gain", "mean"),
        Median_clean_gain=("Clean_mean_gain", "median"),
        Mean_fraction_removed=("Mean_fraction_removed", "mean"),
        Median_fraction_removed=("Mean_fraction_removed", "median"),
    )
    .reset_index()
)

participant_qc_file = (
    OUT / "Wu_binocular_pursuit_cleaning_participant_QC.csv"
)

participant_qc.to_csv(
    participant_qc_file,
    index=False,
)

print("\nSaved:")
print(trial_file)
print(participant_qc_file)

Clean valid: 199 / 200

Fraction removed by group:
           mean  median   max
Group                        
NC     0.080560     0.0  0.45
PD     0.072087     0.0  0.40

Saved:
/content/drive/MyDrive/Wu_PD/analysis_output/Wu_binocular_pursuit_cleaning_QC.csv
/content/drive/MyDrive/Wu_PD/analysis_output/Wu_binocular_pursuit_cleaning_participant_QC.csv


## 5. Primary participant-level variability analysis

Participant is the experimental unit.

- **Primary:** IQR
- **Sensitivity:** MAD and SD
- two-sided Mann–Whitney U
- Cliff's delta
- exact participant-label permutation for IQR
- participant bootstrap CI for the PD–NC median-IQR difference
- leave-one-participant-out robustness

In [6]:
# 5. PRIMARY VARIABILITY ANALYSIS

def participant_summary(trial_data):
    d = trial_data[
        trial_data["Clean_mean_gain"].notna()
    ].copy()

    out = (
        d.groupby(["Participant", "Group"])
        .agg(
            N_trials=("Clean_mean_gain", "count"),
            Mean_gain=("Clean_mean_gain", "mean"),
            Median_gain=("Clean_mean_gain", "median"),
            SD_gain=("Clean_mean_gain", "std"),
            Q25=(
                "Clean_mean_gain",
                lambda x: np.nanpercentile(x, 25),
            ),
            Q75=(
                "Clean_mean_gain",
                lambda x: np.nanpercentile(x, 75),
            ),
            MAD_gain=("Clean_mean_gain", mad_unscaled),
        )
        .reset_index()
    )

    out["IQR_gain"] = out["Q75"] - out["Q25"]
    return out


def exact_permutation_p(df, metric="IQR_gain"):
    values = df[metric].to_numpy()
    groups = df["Group"].to_numpy()

    observed = (
        np.median(values[groups == "PD"])
        - np.median(values[groups == "NC"])
    )

    n_total = len(values)
    n_pd = np.sum(groups == "PD")
    all_idx = np.arange(n_total)

    extreme = 0
    total = 0

    for pd_idx_tuple in combinations(all_idx, n_pd):
        pd_idx = np.asarray(pd_idx_tuple)
        mask = np.ones(n_total, dtype=bool)
        mask[pd_idx] = False
        nc_idx = all_idx[mask]

        difference = (
            np.median(values[pd_idx])
            - np.median(values[nc_idx])
        )

        extreme += abs(difference) >= abs(observed)
        total += 1

    return extreme / total


def bootstrap_group_difference(
    df,
    metric="IQR_gain",
    B=20000,
    seed=12345,
):
    rng = np.random.default_rng(seed)

    nc = df.loc[df["Group"] == "NC", metric].to_numpy()
    pdv = df.loc[df["Group"] == "PD", metric].to_numpy()

    boot = np.empty(B)

    for b in range(B):
        nc_b = rng.choice(nc, len(nc), replace=True)
        pd_b = rng.choice(pdv, len(pdv), replace=True)

        boot[b] = (
            np.median(pd_b)
            - np.median(nc_b)
        )

    return boot


participant = participant_summary(trial_df)

print(
    participant[
        [
            "Participant",
            "Group",
            "N_trials",
            "Mean_gain",
            "Median_gain",
            "IQR_gain",
            "MAD_gain",
            "SD_gain",
        ]
    ].to_string(index=False)
)

result_rows = []

for metric in ["IQR_gain", "MAD_gain", "SD_gain"]:
    nc = participant.loc[
        participant["Group"] == "NC",
        metric,
    ].to_numpy()

    pdv = participant.loc[
        participant["Group"] == "PD",
        metric,
    ].to_numpy()

    u, p = mannwhitneyu(
        pdv,
        nc,
        alternative="two-sided",
    )

    result_rows.append({
        "Metric": metric,
        "NC_median": np.median(nc),
        "PD_median": np.median(pdv),
        "PD_minus_NC": np.median(pdv) - np.median(nc),
        "Mann_Whitney_U": u,
        "Mann_Whitney_p": p,
        "Cliffs_delta": cliffs_delta(pdv, nc),
    })

results_df = pd.DataFrame(result_rows)

perm_p = exact_permutation_p(
    participant,
    "IQR_gain",
)

bootstrap_diff = bootstrap_group_difference(
    participant,
    "IQR_gain",
    B=20000,
    seed=12345,
)

bootstrap_ci = np.percentile(
    bootstrap_diff,
    [2.5, 97.5],
)

primary_difference = (
    participant.loc[
        participant["Group"] == "PD",
        "IQR_gain",
    ].median()
    - participant.loc[
        participant["Group"] == "NC",
        "IQR_gain",
    ].median()
)

print("\nPRIMARY IQR RESULT")
print("PD - NC median difference:", primary_difference)
print("Exact permutation p:", perm_p)
print("Participant-bootstrap 95% CI:", bootstrap_ci)
print("\nAll metrics:")
print(results_df.to_string(index=False))

loo_rows = []

for excluded in participant["Participant"]:
    temp = participant[
        participant["Participant"] != excluded
    ]

    nc = temp.loc[temp["Group"] == "NC", "IQR_gain"].to_numpy()
    pdv = temp.loc[temp["Group"] == "PD", "IQR_gain"].to_numpy()

    u, p = mannwhitneyu(
        pdv,
        nc,
        alternative="two-sided",
    )

    loo_rows.append({
        "Excluded": excluded,
        "PD_minus_NC": np.median(pdv) - np.median(nc),
        "Mann_Whitney_p": p,
    })

loo = pd.DataFrame(loo_rows)

print("\nLOO difference range:")
print(loo["PD_minus_NC"].min(), "to", loo["PD_minus_NC"].max())

print("\nLOO p-value range:")
print(loo["Mann_Whitney_p"].min(), "to", loo["Mann_Whitney_p"].max())

participant_file = (
    OUT / "Wu_binocular_pursuit_variability_participant.csv"
)
results_file = (
    OUT / "Wu_binocular_pursuit_variability_results.csv"
)
loo_file = (
    OUT / "Wu_binocular_pursuit_variability_LOO.csv"
)

participant.to_csv(participant_file, index=False)
results_df.to_csv(results_file, index=False)
loo.to_csv(loo_file, index=False)

print("\nSaved:")
print(participant_file)
print(results_file)
print(loo_file)

Participant Group  N_trials  Mean_gain  Median_gain  IQR_gain  MAD_gain  SD_gain
        NC1    NC        10   0.960686     0.921987  0.201126  0.107207 0.165225
       NC10    NC        10   0.920165     0.922381  0.038404  0.022082 0.052091
        NC2    NC        10   1.055329     1.021842  0.070730  0.030930 0.134249
        NC3    NC        10   0.818690     0.913041  0.351237  0.092285 0.281437
        NC4    NC        10   1.149955     1.131285  0.262303  0.133507 0.211964
        NC5    NC        10   1.115786     1.097928  0.342734  0.189048 0.300800
        NC6    NC        10   1.009415     1.067907  0.158255  0.070143 0.350272
        NC7    NC        10   0.971847     0.938271  0.126658  0.058555 0.107820
        NC8    NC        10   0.908318     0.949157  0.298069  0.150633 0.195857
        NC9    NC        10   0.986169     0.978417  0.059971  0.035189 0.051059
        PD1    PD        10   0.843925     0.911777  0.320525  0.149511 0.248480
       PD10    PD        10 

## 6. Adjustment for typical pursuit gain

HC3-robust OLS tests whether the PD coefficient for variability changes after adjustment for participant median gain. Mean gain is used as a sensitivity covariate.

In [7]:
# 6. TYPICAL-GAIN ADJUSTMENT

adjustment_df = participant.copy()

adjustment_df["Group"] = pd.Categorical(
    adjustment_df["Group"],
    categories=["NC", "PD"],
)

print("Association between median gain and variability:")

for metric in ["IQR_gain", "MAD_gain", "SD_gain"]:
    rho, p = spearmanr(
        adjustment_df["Median_gain"],
        adjustment_df[metric],
    )
    print(
        f"{metric}: Spearman rho={rho:.4f}, p={p:.6f}"
    )


def adjusted_models(covariate):
    rows = []

    for metric in ["IQR_gain", "MAD_gain", "SD_gain"]:
        model = smf.ols(
            f"{metric} ~ {covariate} + C(Group)",
            data=adjustment_df,
        ).fit(cov_type="HC3")

        term = "C(Group)[T.PD]"
        ci = model.conf_int().loc[term]

        rows.append({
            "Outcome": metric,
            "Covariate": covariate,
            "PD_coefficient": model.params[term],
            "SE": model.bse[term],
            "p": model.pvalues[term],
            "CI_low": ci.iloc[0],
            "CI_high": ci.iloc[1],
        })

    return pd.DataFrame(rows)


median_adjusted = adjusted_models("Median_gain")
mean_adjusted = adjusted_models("Mean_gain")

print("\nAdjusted for median gain:")
print(median_adjusted.to_string(index=False))

print("\nAdjusted for mean gain:")
print(mean_adjusted.to_string(index=False))

median_adjusted.to_csv(
    OUT / "Wu_pursuit_variability_adjusted_for_median_gain.csv",
    index=False,
)

mean_adjusted.to_csv(
    OUT / "Wu_pursuit_variability_adjusted_for_mean_gain.csv",
    index=False,
)

adjustment_df.to_csv(
    OUT / "Wu_pursuit_variability_adjusted_participants.csv",
    index=False,
)

Association between median gain and variability:
IQR_gain: Spearman rho=0.0015, p=0.994980
MAD_gain: Spearman rho=0.3068, p=0.188323
SD_gain: Spearman rho=0.2165, p=0.359146

Adjusted for median gain:
 Outcome   Covariate  PD_coefficient       SE        p    CI_low  CI_high
IQR_gain Median_gain        0.034258 0.044711 0.443549 -0.053374 0.121890
MAD_gain Median_gain        0.038231 0.022478 0.088974 -0.005825 0.082286
 SD_gain Median_gain        0.067093 0.045404 0.139490 -0.021897 0.156084

Adjusted for mean gain:
 Outcome Covariate  PD_coefficient       SE        p    CI_low  CI_high
IQR_gain Mean_gain        0.029761 0.045033 0.508693 -0.058502 0.118025
MAD_gain Mean_gain        0.038737 0.021922 0.077228 -0.004230 0.081704
 SD_gain Mean_gain        0.066468 0.047446 0.161241 -0.026525 0.159460


## 7. Validation against Wu et al. supplementary pursuit velocity

Participant mean cleaned gain is converted to velocity by multiplying by the 10°/s target speed. Correspondence with Wu et al.'s supplementary participant-level pursuit velocity is assessed using Pearson correlation, Spearman correlation, MAE, RMSE and Bland–Altman bias/limits of agreement.

The reconstruction is treated as a **validated approximation**, not an exact replication of Wu et al.'s original preprocessing.

In [9]:
# ============================================================
# 7. PIPELINE VALIDATION AGAINST WU SUPPLEMENT
# ============================================================

# Find Wu supplementary XLSX
xlsx_candidates = list(
    BASE.rglob("peerj-06-5442-s002.xlsx")
)

if not xlsx_candidates:
    xlsx_candidates = [
        p for p in BASE.rglob("*.xlsx")
        if "5442" in p.name.lower()
        or "s002" in p.name.lower()
    ]

if not xlsx_candidates:
    raise FileNotFoundError(
        "Could not find peerj-06-5442-s002.xlsx in Wu_PD."
    )

WU_FILE = xlsx_candidates[0]

print("Using Wu supplement:")
print(WU_FILE)

# ------------------------------------------------------------
# Read the actual pursuit-velocity worksheet
# ------------------------------------------------------------

wu = pd.read_excel(
    WU_FILE,
    sheet_name="PursuitVelocity"
)

print("\nWu pursuit sheet:")
print(wu.head())
print("\nColumns:", list(wu.columns))

# The supplement contains:
#   Type = PD / NC / YC
#   Velo = participant-level pursuit velocity
#
# There is no participant-ID column, so IDs are reconstructed
# from row order within each group.

wu = wu.rename(
    columns={
        "Type": "Group",
        "Velo": "Wu_velocity",
    }
)

wu["Group"] = (
    wu["Group"]
    .astype(str)
    .str.strip()
    .str.upper()
)

wu["Wu_velocity"] = pd.to_numeric(
    wu["Wu_velocity"],
    errors="coerce",
)

wu = wu[
    wu["Group"].isin(["PD", "NC", "YC"])
    & wu["Wu_velocity"].notna()
].copy()

# ------------------------------------------------------------
# Reconstruct participant labels from within-group row order
# ------------------------------------------------------------

wu["Participant_number"] = (
    wu.groupby("Group")
    .cumcount()
    + 1
)

wu["Participant"] = (
    wu["Group"]
    + wu["Participant_number"].astype(str)
)

print("\nWu participant-level values:")
print(
    wu[
        [
            "Participant",
            "Group",
            "Wu_velocity",
        ]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# Our reconstructed participant-level values
# ------------------------------------------------------------

our = participant.copy()

our["Our_mean_velocity"] = (
    our["Mean_gain"]
    * TARGET_SPEED
)

our["Our_median_velocity"] = (
    our["Median_gain"]
    * TARGET_SPEED
)

# ------------------------------------------------------------
# Merge PD + NC participants
# ------------------------------------------------------------

validation = our.merge(
    wu[
        [
            "Participant",
            "Group",
            "Wu_velocity",
        ]
    ],
    on=[
        "Participant",
        "Group",
    ],
    how="left",
)

print("\nMatched participant validation table:")
print(
    validation[
        [
            "Participant",
            "Group",
            "Our_mean_velocity",
            "Wu_velocity",
        ]
    ].to_string(index=False)
)

if validation["Wu_velocity"].notna().sum() != 20:
    raise ValueError(
        "Expected 20 matched PD/NC participants, "
        f"but found {validation['Wu_velocity'].notna().sum()}."
    )

# ------------------------------------------------------------
# Agreement metrics
# ------------------------------------------------------------

x = validation["Wu_velocity"].to_numpy()
y = validation["Our_mean_velocity"].to_numpy()

pearson_r, pearson_p = pearsonr(
    x,
    y,
)

spearman_rho, spearman_p = spearmanr(
    x,
    y,
)

difference = (
    y
    - x
)

mae = np.mean(
    np.abs(difference)
)

rmse = np.sqrt(
    np.mean(
        difference ** 2
    )
)

bias = np.mean(
    difference
)

sd_difference = np.std(
    difference,
    ddof=1,
)

loa_low = (
    bias
    - 1.96 * sd_difference
)

loa_high = (
    bias
    + 1.96 * sd_difference
)

validation_metrics = pd.DataFrame(
    [
        {
            "N": len(validation),
            "Pearson_r": pearson_r,
            "Pearson_p": pearson_p,
            "Spearman_rho": spearman_rho,
            "Spearman_p": spearman_p,
            "MAE_deg_s": mae,
            "RMSE_deg_s": rmse,
            "Bias_deg_s": bias,
            "LOA_low_deg_s": loa_low,
            "LOA_high_deg_s": loa_high,
        }
    ]
)

print("\n" + "=" * 80)
print("VALIDATION METRICS")
print("=" * 80)

print(
    validation_metrics.to_string(
        index=False
    )
)

# ------------------------------------------------------------
# Group-level comparison
# ------------------------------------------------------------

wu_pd = validation.loc[
    validation["Group"] == "PD",
    "Wu_velocity",
]

wu_nc = validation.loc[
    validation["Group"] == "NC",
    "Wu_velocity",
]

our_pd = validation.loc[
    validation["Group"] == "PD",
    "Our_mean_velocity",
]

our_nc = validation.loc[
    validation["Group"] == "NC",
    "Our_mean_velocity",
]

print("\n" + "=" * 80)
print("WU VALUES: PD vs NC")
print("=" * 80)

print(
    "PD mean:",
    wu_pd.mean()
)

print(
    "NC mean:",
    wu_nc.mean()
)

u_wu, p_wu = mannwhitneyu(
    wu_pd,
    wu_nc,
    alternative="two-sided",
)

t_wu, pt_wu = ttest_ind(
    wu_pd,
    wu_nc,
    equal_var=False,
)

print(
    f"Mann-Whitney U = {u_wu:.3f}, "
    f"p = {p_wu:.6f}"
)

print(
    f"Welch t = {t_wu:.3f}, "
    f"p = {pt_wu:.6f}"
)

print("\n" + "=" * 80)
print("OUR CLEANED VALUES: PD vs NC")
print("=" * 80)

print(
    "PD mean:",
    our_pd.mean()
)

print(
    "NC mean:",
    our_nc.mean()
)

u_our, p_our = mannwhitneyu(
    our_pd,
    our_nc,
    alternative="two-sided",
)

t_our, pt_our = ttest_ind(
    our_pd,
    our_nc,
    equal_var=False,
)

print(
    f"Mann-Whitney U = {u_our:.3f}, "
    f"p = {p_our:.6f}"
)

print(
    f"Welch t = {t_our:.3f}, "
    f"p = {pt_our:.6f}"
)

# ------------------------------------------------------------
# Three-group Wu supplement ANOVA + Tukey
# ------------------------------------------------------------

wu_three = wu[
    wu["Group"].isin(
        [
            "NC",
            "PD",
            "YC",
        ]
    )
].copy()

anova_model = smf.ols(
    "Wu_velocity ~ C(Group)",
    data=wu_three,
).fit()

anova_table = sm.stats.anova_lm(
    anova_model,
    typ=2,
)

print("\n" + "=" * 80)
print("WU SUPPLEMENT: THREE-GROUP ANOVA")
print("=" * 80)

print(
    anova_table
)

print("\n" + "=" * 80)
print("WU SUPPLEMENT: TUKEY HSD")
print("=" * 80)

print(
    pairwise_tukeyhsd(
        endog=wu_three["Wu_velocity"],
        groups=wu_three["Group"],
        alpha=0.05,
    )
)

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

validation.to_csv(
    OUT / "Wu_pipeline_validation_participants.csv",
    index=False,
)

validation_metrics.to_csv(
    OUT / "Wu_pipeline_validation_metrics.csv",
    index=False,
)

print("\nSaved:")
print(
    OUT
    / "Wu_pipeline_validation_participants.csv"
)

print(
    OUT
    / "Wu_pipeline_validation_metrics.csv"
)

Using Wu supplement:
/content/drive/MyDrive/Wu_PD/peerj-06-5442-s002.xlsx

Wu pursuit sheet:
  Type     Velo
0   PD   8.0603
1   PD  10.5699
2   PD  10.3060
3   PD   9.6272
4   PD   9.9303

Columns: ['Type', 'Velo']

Wu participant-level values:
Participant Group  Wu_velocity
        PD1    PD       8.0603
        PD2    PD      10.5699
        PD3    PD      10.3060
        PD4    PD       9.6272
        PD5    PD       9.9303
        PD6    PD       8.4840
        PD7    PD       5.8927
        PD8    PD       8.3783
        PD9    PD       8.8603
       PD10    PD       9.3600
        NC1    NC       9.8236
        NC2    NC      10.9355
        NC3    NC       8.1081
        NC4    NC      10.1749
        NC5    NC      10.2153
        NC6    NC      10.1188
        NC7    NC       9.7474
        NC8    NC       8.4804
        NC9    NC      10.6419
       NC10    NC       9.4029
        YC1    YC      11.3718
        YC2    YC      10.7906
        YC3    YC      11.0867
        YC

## 8. Cleaning sensitivity

The entire primary IQR analysis is repeated for 12 combinations of absolute velocity limit (25, 30, 35, 40°/s) and robust threshold (4, 6, 8 robust SD). The primary specification remains 30°/s + 6 robust SD.

In [ ]:
# 8. CLEANING-SENSITIVITY ANALYSIS

ABS_LIMITS = [25.0, 30.0, 35.0, 40.0]
ROBUST_K_VALUES = [4.0, 6.0, 8.0]
BOOTSTRAPS = 10000
RNG_SEED = 12345

sensitivity_rows = []
sensitivity_participants = []
sensitivity_trials = []

for abs_limit in ABS_LIMITS:
    for robust_k in ROBUST_K_VALUES:
        print(
            f"Processing {abs_limit:g} deg/s + "
            f"{robust_k:g} robust-SD"
        )

        temp_trials = process_specification(
            raw_trials,
            abs_limit,
            robust_k,
        )

        temp_participant = participant_summary(
            temp_trials
        )

        nc = temp_participant.loc[
            temp_participant["Group"] == "NC",
            "IQR_gain",
        ].to_numpy()

        pdv = temp_participant.loc[
            temp_participant["Group"] == "PD",
            "IQR_gain",
        ].to_numpy()

        observed = np.median(pdv) - np.median(nc)

        u, mw_p = mannwhitneyu(
            pdv,
            nc,
            alternative="two-sided",
        )

        exact_p = exact_permutation_p(
            temp_participant,
            "IQR_gain",
        )

        seed = RNG_SEED + int(abs_limit * 10 + robust_k)

        boot = bootstrap_group_difference(
            temp_participant,
            "IQR_gain",
            B=BOOTSTRAPS,
            seed=seed,
        )

        ci_low, ci_high = np.percentile(
            boot,
            [2.5, 97.5],
        )

        sensitivity_rows.append({
            "Abs_limit": abs_limit,
            "Robust_k": robust_k,
            "Primary_specification": (
                abs_limit == PRIMARY_ABS_LIMIT
                and robust_k == PRIMARY_ROBUST_K
            ),
            "Valid_trials": int(
                temp_trials["Clean_mean_gain"].notna().sum()
            ),
            "NC_median_IQR": np.median(nc),
            "PD_median_IQR": np.median(pdv),
            "PD_minus_NC": observed,
            "Mann_Whitney_U": u,
            "Mann_Whitney_p": mw_p,
            "Exact_permutation_p": exact_p,
            "Cliffs_delta": cliffs_delta(pdv, nc),
            "Bootstrap_CI_low": ci_low,
            "Bootstrap_CI_high": ci_high,
        })

        pcopy = temp_participant.copy()
        pcopy["Abs_limit"] = abs_limit
        pcopy["Robust_k"] = robust_k
        sensitivity_participants.append(pcopy)

        tcopy = temp_trials.copy()
        tcopy["Abs_limit"] = abs_limit
        tcopy["Robust_k"] = robust_k
        sensitivity_trials.append(tcopy)

sensitivity = pd.DataFrame(sensitivity_rows)

sensitivity_participants = pd.concat(
    sensitivity_participants,
    ignore_index=True,
)

sensitivity_trials = pd.concat(
    sensitivity_trials,
    ignore_index=True,
)

print(
    sensitivity[
        [
            "Abs_limit",
            "Robust_k",
            "PD_minus_NC",
            "Mann_Whitney_p",
            "Exact_permutation_p",
            "Cliffs_delta",
            "Bootstrap_CI_low",
            "Bootstrap_CI_high",
            "Valid_trials",
        ]
    ].to_string(index=False)
)

sensitivity.to_csv(
    OUT / "Wu_cleaning_sensitivity_summary.csv",
    index=False,
)

sensitivity_participants.to_csv(
    OUT / "Wu_cleaning_sensitivity_participants.csv",
    index=False,
)

sensitivity_trials.to_csv(
    OUT / "Wu_cleaning_sensitivity_trials.csv",
    index=False,
)

## 9. Finite-trial uncertainty

Trials are resampled **within participant** with replacement to quantify uncertainty in participant IQR estimates caused by having only 9–10 valid binocular trials. This is separate from the participant bootstrap used for population-sampling uncertainty.

In [ ]:
# 9. WITHIN-PARTICIPANT TRIAL RESAMPLING

RNG_SEED_TRIALS = 12345
N_BOOT_PARTICIPANT = 20000
N_BOOT_GROUP = 20000

rng = np.random.default_rng(RNG_SEED_TRIALS)

participant_uncertainty_rows = []

for participant_id in participant["Participant"]:
    values = valid_trials.loc[
        valid_trials["Participant"] == participant_id,
        "Clean_mean_gain",
    ].to_numpy()

    group = valid_trials.loc[
        valid_trials["Participant"] == participant_id,
        "Group",
    ].iloc[0]

    observed_iqr = iqr_value(values)

    boot_iqr = np.empty(N_BOOT_PARTICIPANT)

    for b in range(N_BOOT_PARTICIPANT):
        sample = rng.choice(
            values,
            size=len(values),
            replace=True,
        )
        boot_iqr[b] = iqr_value(sample)

    ci_low, ci_high = np.percentile(
        boot_iqr,
        [2.5, 97.5],
    )

    participant_uncertainty_rows.append({
        "Participant": participant_id,
        "Group": group,
        "N_trials": len(values),
        "Observed_IQR": observed_iqr,
        "Bootstrap_median_IQR": np.median(boot_iqr),
        "Bootstrap_SD_IQR": np.std(boot_iqr, ddof=1),
        "CI_low": ci_low,
        "CI_high": ci_high,
        "CI_width": ci_high - ci_low,
    })

participant_uncertainty = pd.DataFrame(
    participant_uncertainty_rows
)

print("PARTICIPANT IQR UNCERTAINTY")
print(participant_uncertainty.to_string(index=False))

print("\nUNCERTAINTY WIDTH BY GROUP")
print(
    participant_uncertainty.groupby("Group")[
        ["Bootstrap_SD_IQR", "CI_width"]
    ].agg(["mean", "median", "std", "min", "max"])
)

participant_ids = participant["Participant"].to_numpy()
participant_groups = participant["Group"].to_numpy()

group_boot = np.empty(N_BOOT_GROUP)

for b in range(N_BOOT_GROUP):
    simulated_iqrs = []

    for participant_id in participant_ids:
        values = valid_trials.loc[
            valid_trials["Participant"] == participant_id,
            "Clean_mean_gain",
        ].to_numpy()

        sample = rng.choice(
            values,
            size=len(values),
            replace=True,
        )

        simulated_iqrs.append(iqr_value(sample))

    simulated_iqrs = np.asarray(simulated_iqrs)

    group_boot[b] = (
        np.median(simulated_iqrs[participant_groups == "PD"])
        - np.median(simulated_iqrs[participant_groups == "NC"])
    )

trial_resampling_ci = np.percentile(
    group_boot,
    [2.5, 97.5],
)

fraction_positive = np.mean(group_boot > 0)

print("\nGROUP DIFFERENCE UNCERTAINTY FROM LIMITED TRIAL COUNT")
print("Observed difference:", primary_difference)
print("95% resampling interval:", trial_resampling_ci)
print("Fraction > 0:", fraction_positive)

participant_uncertainty.to_csv(
    OUT / "Wu_trial_count_uncertainty_participants.csv",
    index=False,
)

pd.DataFrame([{
    "Trials_per_participant": "observed 9-10",
    "Iterations": N_BOOT_GROUP,
    "Observed_PD_minus_NC": primary_difference,
    "Resampling_CI_low": trial_resampling_ci[0],
    "Resampling_CI_high": trial_resampling_ci[1],
    "Fraction_PD_gt_NC": fraction_positive,
}]).to_csv(
    OUT / "Wu_trial_count_sensitivity.csv",
    index=False,
)

pd.DataFrame({
    "PD_minus_NC_trial_resampling": group_boot
}).to_csv(
    OUT / "Wu_trial_resampling_group_difference.csv",
    index=False,
)

## 10. Trial-order dynamics

A linear trial-order model tests whether pursuit gain changes systematically across the 10-trial binocular block and whether the slope differs by group. The mixed model is attempted first; participant-clustered OLS is retained as the robust fallback/sensitivity model. Participant-specific slopes are also compared nonparametrically.

In [ ]:
# 10. TRIAL-ORDER DYNAMICS

trial_dyn = valid_trials[
    ["Participant", "Group", "Trial", "Clean_mean_gain"]
].copy()

trial_dyn["Trial_centered"] = trial_dyn["Trial"] - 5.5

trial_dyn["Group"] = pd.Categorical(
    trial_dyn["Group"],
    categories=["NC", "PD"],
)

formula = "Clean_mean_gain ~ Trial_centered * C(Group)"

mixed_result = None

try:
    mixed_model = smf.mixedlm(
        formula,
        data=trial_dyn,
        groups=trial_dyn["Participant"],
    )
    mixed_result = mixed_model.fit(
        reml=True,
        method="lbfgs",
    )
    print("MIXED MODEL")
    print(mixed_result.summary())
except Exception as exc:
    print("Mixed model could not be estimated reliably:")
    print(repr(exc))

ols = smf.ols(
    formula,
    data=trial_dyn,
).fit(
    cov_type="cluster",
    cov_kwds={"groups": trial_dyn["Participant"]},
)

print("\nPARTICIPANT-CLUSTERED OLS")
print(ols.summary())

trial_term = "Trial_centered"
interaction_term = "Trial_centered:C(Group)[T.PD]"

nc_slope_model = ols.params[trial_term]
interaction_beta = ols.params[interaction_term]
pd_slope_model = nc_slope_model + interaction_beta
interaction_ci = ols.conf_int().loc[interaction_term]

print("\nKEY RESULTS")
print(
    f"NC slope = {nc_slope_model:.6f}, "
    f"p = {ols.pvalues[trial_term]:.6f}"
)
print(
    f"Trial x PD interaction beta = {interaction_beta:.6f}, "
    f"95% CI [{interaction_ci.iloc[0]:.6f}, "
    f"{interaction_ci.iloc[1]:.6f}], "
    f"p = {ols.pvalues[interaction_term]:.6f}"
)
print(f"Implied PD slope = {pd_slope_model:.6f}")

slope_rows = []

for (participant_id, group), sub in trial_dyn.groupby(
    ["Participant", "Group"],
    observed=True,
):
    x = sub["Trial"].to_numpy(dtype=float)
    y = sub["Clean_mean_gain"].to_numpy(dtype=float)

    slope, intercept = np.polyfit(x, y, 1)

    slope_rows.append({
        "Participant": participant_id,
        "Group": str(group),
        "Slope_gain_per_trial": slope,
        "Intercept": intercept,
        "N_trials": len(sub),
    })

slopes = pd.DataFrame(slope_rows)

nc_slopes = slopes.loc[
    slopes["Group"] == "NC",
    "Slope_gain_per_trial",
].to_numpy()

pd_slopes = slopes.loc[
    slopes["Group"] == "PD",
    "Slope_gain_per_trial",
].to_numpy()

u_slope, p_slope = mannwhitneyu(
    pd_slopes,
    nc_slopes,
    alternative="two-sided",
)

delta_slope = cliffs_delta(pd_slopes, nc_slopes)

rng_slopes = np.random.default_rng(12345)
B_SLOPE = 20000
slope_boot = np.empty(B_SLOPE)

for b in range(B_SLOPE):
    nc_b = rng_slopes.choice(
        nc_slopes,
        len(nc_slopes),
        replace=True,
    )
    pd_b = rng_slopes.choice(
        pd_slopes,
        len(pd_slopes),
        replace=True,
    )

    slope_boot[b] = np.median(pd_b) - np.median(nc_b)

slope_ci = np.percentile(
    slope_boot,
    [2.5, 97.5],
)

print("\nPARTICIPANT-SLOPE COMPARISON")
print("NC median slope:", np.median(nc_slopes))
print("PD median slope:", np.median(pd_slopes))
print("PD - NC median slope:", np.median(pd_slopes) - np.median(nc_slopes))
print("Mann-Whitney U:", u_slope)
print("p:", p_slope)
print("Cliff delta:", delta_slope)
print("Bootstrap 95% CI:", slope_ci)

trial_dyn.to_csv(
    OUT / "Wu_trial_dynamics_by_trial.csv",
    index=False,
)

slopes.to_csv(
    OUT / "Wu_trial_dynamics_participant_slopes.csv",
    index=False,
)

pd.DataFrame({
    "Term": ols.params.index,
    "Coefficient": ols.params.values,
    "SE": ols.bse.values,
    "p": ols.pvalues.values,
    "CI_low": ols.conf_int()[0].values,
    "CI_high": ols.conf_int()[1].values,
}).to_csv(
    OUT / "Wu_trial_dynamics_clustered_OLS.csv",
    index=False,
)

# 11. Final figures
Only the final manuscript-facing figure versions are generated below. PNG files are saved at 600 dpi and PDF/SVG versions are saved as vector graphics.

In [ ]:
# 11A. FIGURE STYLE

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "axes.linewidth": 0.8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "figure.dpi": 150,
    "savefig.dpi": 600,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.04,
})


def save_figure(fig, stem):
    for ext in ["png", "pdf", "svg"]:
        path = FIGDIR / f"{stem}.{ext}"

        if ext == "png":
            fig.savefig(
                path,
                dpi=600,
                bbox_inches="tight",
            )
        else:
            fig.savefig(
                path,
                bbox_inches="tight",
            )

        print("Saved:", path)


def despine(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def panel_label(ax, label):
    ax.text(
        -0.12,
        1.05,
        label,
        transform=ax.transAxes,
        fontsize=11,
        fontweight="bold",
        va="bottom",
        ha="left",
    )

In [ ]:
# 11B. MAIN FIGURE 1 — PRIMARY ESTIMATION

nc_iqr = participant.loc[
    participant["Group"] == "NC",
    "IQR_gain",
].to_numpy()

pd_iqr = participant.loc[
    participant["Group"] == "PD",
    "IQR_gain",
].to_numpy()

rng_group_ci = np.random.default_rng(20260913)

def bootstrap_median_ci(values, B=50000):
    boots = np.empty(B)
    values = np.asarray(values)

    for b in range(B):
        sample = rng_group_ci.choice(
            values,
            len(values),
            replace=True,
        )
        boots[b] = np.median(sample)

    return np.percentile(boots, [2.5, 97.5])


nc_ci = bootstrap_median_ci(nc_iqr)
pd_ci = bootstrap_median_ci(pd_iqr)

rng_effect = np.random.default_rng(20260913)
B = 50000
effect_boot = np.empty(B)

for b in range(B):
    nc_b = rng_effect.choice(
        nc_iqr,
        len(nc_iqr),
        replace=True,
    )
    pd_b = rng_effect.choice(
        pd_iqr,
        len(pd_iqr),
        replace=True,
    )
    effect_boot[b] = np.median(pd_b) - np.median(nc_b)

effect_ci = np.percentile(
    effect_boot,
    [2.5, 97.5],
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(6.8, 3.2),
    gridspec_kw={"width_ratios": [1.15, 1]},
)

ax = axes[0]

for x, vals in [(0, nc_iqr), (1, pd_iqr)]:
    jitter = np.linspace(-0.055, 0.055, len(vals))

    ax.scatter(
        x + jitter,
        vals,
        facecolors="white",
        edgecolors="black",
        s=28,
        linewidth=0.9,
        zorder=3,
    )

for x, vals, ci in [
    (0, nc_iqr, nc_ci),
    (1, pd_iqr, pd_ci),
]:
    med = np.median(vals)

    ax.errorbar(
        x,
        med,
        yerr=[[med - ci[0]], [ci[1] - med]],
        fmt="s",
        markersize=6,
        capsize=3,
        color="black",
        zorder=4,
    )

ax.set_xticks([0, 1], ["NC", "PD"])
ax.set_ylabel("Trial-to-trial pursuit gain IQR")
ax.set_xlabel("Group")
despine(ax)
panel_label(ax, "A")

ax = axes[1]

ax.axvline(
    0,
    linestyle="--",
    linewidth=0.9,
    color="0.65",
)

ax.errorbar(
    primary_difference,
    0,
    xerr=[
        [primary_difference - effect_ci[0]],
        [effect_ci[1] - primary_difference],
    ],
    fmt="D",
    markersize=6,
    capsize=5,
    color="black",
)

ax.set_yticks([])
ax.set_xlabel("PD − NC median IQR difference")
ax.text(
    0.05,
    0.90,
    f"Difference = {primary_difference:.3f}\n"
    f"95% CI [{effect_ci[0]:.3f}, {effect_ci[1]:.3f}]",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=8,
)

despine(ax)
ax.spines["left"].set_visible(False)
panel_label(ax, "B")

fig.tight_layout()

save_figure(
    fig,
    "Figure_1_primary_estimation",
)

plt.show()

In [ ]:
# 11C. MAIN FIGURE 2 — CLEANING ROBUSTNESS

sens_plot = sensitivity.sort_values(
    ["Abs_limit", "Robust_k"]
).reset_index(drop=True)

labels = [
    f"{row.Abs_limit:g}°/s + {row.Robust_k:g} robust-SD"
    for row in sens_plot.itertuples()
]

fig, ax = plt.subplots(figsize=(6.4, 4.8))

ax.axvline(
    0,
    linestyle="--",
    linewidth=0.9,
    color="0.65",
)

for i, row in sens_plot.iterrows():
    primary_here = (
        row["Abs_limit"] == PRIMARY_ABS_LIMIT
        and row["Robust_k"] == PRIMARY_ROBUST_K
    )

    ax.errorbar(
        row["PD_minus_NC"],
        i,
        xerr=[[
            row["PD_minus_NC"] - row["Bootstrap_CI_low"]
        ], [
            row["Bootstrap_CI_high"] - row["PD_minus_NC"]
        ]],
        fmt="D" if primary_here else "o",
        markersize=5.5 if primary_here else 4.5,
        markerfacecolor="black" if primary_here else "white",
        markeredgecolor="black",
        color="black",
        linewidth=1,
        capsize=0,
    )

ax.set_yticks(np.arange(len(sens_plot)))
ax.set_yticklabels(labels)
ax.invert_yaxis()
ax.set_xlabel("PD − NC difference in median pursuit-gain IQR")
ax.set_ylabel("Velocity-cleaning specification")
despine(ax)

fig.tight_layout()

save_figure(
    fig,
    "Figure_2_velocity_cleaning_sensitivity",
)

plt.show()

In [ ]:
# 11D. MAIN FIGURE 3 — FINITE-TRIAL UNCERTAINTY

fig = plt.figure(figsize=(7.1, 5.0))

gs = fig.add_gridspec(
    2,
    2,
    height_ratios=[2.2, 1],
)

ax_nc = fig.add_subplot(gs[0, 0])
ax_pd = fig.add_subplot(gs[0, 1])
ax_effect = fig.add_subplot(gs[1, :])

for ax, group, label in [
    (ax_nc, "NC", "A"),
    (ax_pd, "PD", "B"),
]:
    temp = (
        participant_uncertainty[
            participant_uncertainty["Group"] == group
        ]
        .sort_values("Observed_IQR", ascending=False)
        .reset_index(drop=True)
    )

    y = np.arange(len(temp))

    for i, row in temp.iterrows():
        ax.errorbar(
            row["Observed_IQR"],
            i,
            xerr=[[
                row["Observed_IQR"] - row["CI_low"]
            ], [
                row["CI_high"] - row["Observed_IQR"]
            ]],
            fmt="o",
            color="black",
            markersize=4.5,
            linewidth=0.9,
            capsize=0,
        )

    ax.set_yticks(y)
    ax.set_yticklabels(temp["Participant"])
    ax.invert_yaxis()
    ax.set_xlabel("Participant IQR")
    ax.set_title(group, fontweight="bold")
    despine(ax)
    panel_label(ax, label)

ax_nc.set_ylabel("Participant")

ax_effect.axvline(
    0,
    linestyle="--",
    linewidth=0.9,
    color="0.65",
)

ax_effect.errorbar(
    primary_difference,
    0,
    xerr=[[
        primary_difference - trial_resampling_ci[0]
    ], [
        trial_resampling_ci[1] - primary_difference
    ]],
    fmt="D",
    color="black",
    markersize=5.5,
    capsize=5,
)

ax_effect.set_yticks([])
ax_effect.set_xlabel("PD − NC median IQR difference")

ax_effect.text(
    0.02,
    0.88,
    "Trial-resampling 95% interval "
    f"[{trial_resampling_ci[0]:.3f}, {trial_resampling_ci[1]:.3f}]\n"
    f"Positive difference in {100 * fraction_positive:.1f}% of resamples",
    transform=ax_effect.transAxes,
    ha="left",
    va="top",
    fontsize=8,
)

despine(ax_effect)
ax_effect.spines["left"].set_visible(False)
panel_label(ax_effect, "C")

fig.tight_layout()

save_figure(
    fig,
    "Figure_3_measurement_precision",
)

plt.show()

## 12. Supplementary figures

Figure S1 presents trial-order dynamics. Figure S2 presents pipeline validation against Wu et al.'s supplementary pursuit-velocity values.

In [ ]:
# 12A. SUPPLEMENTARY FIGURE S1 — TRIAL ORDER

trial_summary = (
    trial_dyn
    .groupby(["Group", "Trial"], observed=True)
    .agg(
        Mean_gain=("Clean_mean_gain", "mean"),
        SD=("Clean_mean_gain", "std"),
        N=("Clean_mean_gain", "count"),
    )
    .reset_index()
)

trial_summary["SEM"] = (
    trial_summary["SD"] / np.sqrt(trial_summary["N"])
)

fig = plt.figure(figsize=(7.0, 5.0))

gs = fig.add_gridspec(
    2,
    2,
    height_ratios=[1.35, 1],
)

ax_a = fig.add_subplot(gs[0, :])
ax_b = fig.add_subplot(gs[1, 0])
ax_c = fig.add_subplot(gs[1, 1])

for group, marker, linestyle in [
    ("NC", "o", "-"),
    ("PD", "s", "--"),
]:
    temp = trial_summary[
        trial_summary["Group"] == group
    ]

    ax_a.errorbar(
        temp["Trial"],
        temp["Mean_gain"],
        yerr=temp["SEM"],
        marker=marker,
        linestyle=linestyle,
        color="black",
        markerfacecolor="white" if group == "NC" else "black",
        markersize=4.5,
        capsize=2,
        label=group,
    )

ax_a.axhline(
    1.0,
    linestyle=":",
    linewidth=0.8,
    color="0.65",
)
ax_a.set_xlabel("Trial number")
ax_a.set_ylabel("Mean cleaned pursuit gain")
ax_a.legend(frameon=False)
despine(ax_a)
panel_label(ax_a, "A")

for x, group in enumerate(["NC", "PD"]):
    values = slopes.loc[
        slopes["Group"] == group,
        "Slope_gain_per_trial",
    ].to_numpy()

    jitter = np.linspace(-0.055, 0.055, len(values))

    ax_b.scatter(
        x + jitter,
        values,
        facecolors="white",
        edgecolors="black",
        s=26,
    )

    ax_b.plot(
        x,
        np.median(values),
        "_",
        color="black",
        markersize=16,
        markeredgewidth=2,
    )

ax_b.axhline(
    0,
    linestyle="--",
    linewidth=0.8,
    color="0.65",
)
ax_b.set_xticks([0, 1], ["NC", "PD"])
ax_b.set_ylabel("Gain change per trial")
ax_b.set_xlabel("Group")
despine(ax_b)
panel_label(ax_b, "B")

prediction_rows = []

for group in ["NC", "PD"]:
    for trial_number in np.linspace(1, 10, 100):
        prediction_rows.append({
            "Trial": trial_number,
            "Trial_centered": trial_number - 5.5,
            "Group": group,
        })

pred = pd.DataFrame(prediction_rows)

pred["Group"] = pd.Categorical(
    pred["Group"],
    categories=["NC", "PD"],
)

pred["Fitted_gain"] = ols.predict(pred)

for group, linestyle in [
    ("NC", "-"),
    ("PD", "--"),
]:
    temp = pred[pred["Group"] == group]

    ax_c.plot(
        temp["Trial"],
        temp["Fitted_gain"],
        linestyle=linestyle,
        color="black",
        label=group,
    )

ax_c.axhline(
    1.0,
    linestyle=":",
    linewidth=0.8,
    color="0.65",
)
ax_c.set_xlabel("Trial number")
ax_c.set_ylabel("Model-estimated pursuit gain")
despine(ax_c)
panel_label(ax_c, "C")

fig.tight_layout()

save_figure(
    fig,
    "Supplementary_Figure_S1_trial_order_dynamics",
)

plt.show()

print("Figure S1 model values")
print("NC slope:", nc_slope_model)
print("PD slope:", pd_slope_model)
print("Interaction beta:", interaction_beta)
print("Interaction p:", ols.pvalues[interaction_term])

In [ ]:
# 12B. SUPPLEMENTARY FIGURE S2 — PIPELINE VALIDATION

fig, axes = plt.subplots(
    1,
    2,
    figsize=(7.0, 3.2),
)

ax = axes[0]

for group, marker in [
    ("NC", "o"),
    ("PD", "s"),
]:
    temp = validation[
        validation["Group"] == group
    ]

    ax.scatter(
        temp["Wu_velocity"],
        temp["Our_mean_velocity"],
        marker=marker,
        facecolors="white",
        edgecolors="black",
        s=36,
        label=group,
    )

minimum = min(
    validation["Wu_velocity"].min(),
    validation["Our_mean_velocity"].min(),
)

maximum = max(
    validation["Wu_velocity"].max(),
    validation["Our_mean_velocity"].max(),
)

ax.plot(
    [minimum, maximum],
    [minimum, maximum],
    linestyle="--",
    color="0.65",
    linewidth=0.9,
)

ax.set_xlabel("Wu supplementary pursuit velocity (°/s)")
ax.set_ylabel("Reconstructed pursuit velocity (°/s)")
ax.legend(frameon=False)
despine(ax)
panel_label(ax, "A")

ax.text(
    0.04,
    0.96,
    f"r = {pearson_r:.2f}",
    transform=ax.transAxes,
    va="top",
    ha="left",
    fontsize=8,
)

ax = axes[1]

means = (
    validation["Wu_velocity"].to_numpy()
    + validation["Our_mean_velocity"].to_numpy()
) / 2

diffs = (
    validation["Our_mean_velocity"].to_numpy()
    - validation["Wu_velocity"].to_numpy()
)

ax.scatter(
    means,
    diffs,
    facecolors="white",
    edgecolors="black",
    s=34,
)

ax.axhline(
    bias,
    color="black",
    linewidth=1,
)

ax.axhline(
    loa_low,
    color="0.55",
    linestyle="--",
    linewidth=0.9,
)

ax.axhline(
    loa_high,
    color="0.55",
    linestyle="--",
    linewidth=0.9,
)

ax.set_xlabel("Mean velocity (°/s)")
ax.set_ylabel("Reconstructed − Wu (°/s)")
despine(ax)
panel_label(ax, "B")

fig.tight_layout()

save_figure(
    fig,
    "Supplementary_Figure_S2_pipeline_validation",
)

plt.show()

validation.to_csv(
    FIGDIR / "Supplementary_Figure_S2_validation_data.csv",
    index=False,
)

print("Figure S2 validation values")
print("N =", len(validation))
print(f"Pearson r = {pearson_r:.4f}, p = {pearson_p:.6g}")
print(f"Spearman rho = {spearman_rho:.4f}, p = {spearman_p:.6g}")
print(f"MAE = {mae:.4f} deg/s")
print(f"RMSE = {rmse:.4f} deg/s")
print(f"Bias = {bias:.4f} deg/s")
print(
    f"95% limits of agreement = [{loa_low:.4f}, {loa_high:.4f}] deg/s"
)

## 13. Graphical abstract

This cell is separate from the statistical analysis.

In [ ]:
# ============================================================
# 13. GRAPHICAL ABSTRACT
# ============================================================

fig, ax = plt.subplots(figsize=(8.2, 3.4))

ax.set_xlim(0, 14)
ax.set_ylim(0, 6)
ax.axis("off")

ax.text(
    7,
    5.55,
    "Trial-to-trial smooth-pursuit variability in Parkinson’s disease",
    ha="center",
    va="center",
    fontsize=15,
    fontweight="bold",
)

box_y = 1.75
box_h = 2.75

left_x, left_w = 0.45, 3.35
mid_x, mid_w = 5.05, 3.90
right_x, right_w = 10.25, 3.30


def rounded_box(x, y, w, h):
    ax.add_patch(
        FancyBboxPatch(
            (x, y),
            w,
            h,
            boxstyle="round,pad=0.06",
            facecolor="white",
            edgecolor="black",
            linewidth=1.4,
        )
    )


rounded_box(left_x, box_y, left_w, box_h)
rounded_box(mid_x, box_y, mid_w, box_h)
rounded_box(right_x, box_y, right_w, box_h)

ax.text(
    left_x + left_w / 2,
    4.12,
    "Participants",
    ha="center",
    fontsize=11,
    fontweight="bold",
)

ax.text(
    left_x + 1.02,
    3.43,
    "NC",
    ha="center",
    fontsize=15,
    fontweight="bold",
)

ax.text(
    left_x + 2.35,
    3.43,
    "PD",
    ha="center",
    fontsize=15,
    fontweight="bold",
)

ax.text(left_x + 1.02, 3.00, "n = 10", ha="center", fontsize=10)
ax.text(left_x + 2.35, 3.00, "n = 10", ha="center", fontsize=10)

ax.plot(
    [left_x + 0.45, left_x + left_w - 0.45],
    [2.63, 2.63],
    color="0.8",
    linewidth=0.8,
)

ax.text(
    left_x + left_w / 2,
    2.28,
    "10 binocular trials",
    ha="center",
    fontsize=9.5,
)

ax.text(
    left_x + left_w / 2,
    1.98,
    "10°/s target",
    ha="center",
    fontsize=8.5,
)

ax.add_patch(
    FancyArrowPatch(
        (3.95, 3.1),
        (4.85, 3.1),
        arrowstyle="-|>",
        mutation_scale=16,
        linewidth=1.4,
        color="black",
    )
)

ax.text(
    mid_x + mid_w / 2,
    4.12,
    "Pursuit-gain IQR",
    ha="center",
    fontsize=11,
    fontweight="bold",
)

nc_demo = np.array([
    0.48, 0.51, 0.47, 0.52, 0.50,
    0.49, 0.53, 0.46, 0.51, 0.48,
])

pd_demo = np.array([
    0.35, 0.62, 0.43, 0.68, 0.39,
    0.57, 0.31, 0.64, 0.45, 0.59,
])

xpos = np.linspace(
    mid_x + 1.10,
    mid_x + mid_w - 0.35,
    10,
)

ax.text(
    mid_x + 0.68,
    3.42,
    "NC",
    ha="right",
    va="center",
    fontsize=9.5,
    fontweight="bold",
)

ax.scatter(
    xpos,
    3.42 + (nc_demo - 0.5),
    s=28,
    facecolor="white",
    edgecolor="black",
    linewidth=1,
)

ax.text(
    mid_x + 0.68,
    2.70,
    "PD",
    ha="right",
    va="center",
    fontsize=9.5,
    fontweight="bold",
)

ax.scatter(
    xpos,
    2.70 + (pd_demo - 0.5),
    s=28,
    facecolor="black",
    edgecolor="black",
    linewidth=1,
)

ax.text(
    mid_x + mid_w / 2,
    2.05,
    "Median IQR",
    ha="center",
    fontsize=8.5,
)

ax.text(
    mid_x + mid_w / 2,
    1.78,
    "0.180  →  0.216",
    ha="center",
    fontsize=11,
    fontweight="bold",
)

ax.add_patch(
    FancyArrowPatch(
        (9.10, 3.1),
        (10.05, 3.1),
        arrowstyle="-|>",
        mutation_scale=16,
        linewidth=1.4,
        color="black",
    )
)

ax.text(
    right_x + right_w / 2,
    4.12,
    "PD − NC",
    ha="center",
    fontsize=11,
    fontweight="bold",
)

ax.text(
    right_x + right_w / 2,
    3.53,
    "+0.036",
    ha="center",
    fontsize=19,
    fontweight="bold",
)

ax.text(
    right_x + right_w / 2,
    3.06,
    "95% CI",
    ha="center",
    fontsize=8.5,
)

ax.text(
    right_x + right_w / 2,
    2.68,
    "−0.094 to 0.154",
    ha="center",
    fontsize=11,
)

ax.plot(
    [right_x + 0.50, right_x + right_w - 0.50],
    [2.30, 2.30],
    color="0.8",
    linewidth=0.8,
)

ax.text(
    right_x + right_w / 2,
    2.00,
    "Positive direction",
    ha="center",
    fontsize=8.5,
)

ax.text(
    right_x + right_w / 2,
    1.78,
    "but uncertain estimate",
    ha="center",
    fontsize=8.5,
    fontweight="bold",
)

ax.text(
    7,
    0.83,
    "Pursuit stability warrants prospective testing with larger repeated-trial datasets",
    ha="center",
    va="center",
    fontsize=10.5,
    fontweight="bold",
)

save_figure(
    fig,
    "graphical_abstract_final",
)

plt.show()

## 14. Software versions and final output check

Run this cell after the notebook completes.

In [ ]:
# 14. REPRODUCIBILITY SUMMARY

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("statsmodels:", statsmodels.__version__)
print("Matplotlib:", matplotlib.__version__)

expected_outputs = [
    OUT / "Wu_binocular_pursuit_cleaning_QC.csv",
    OUT / "Wu_binocular_pursuit_cleaning_participant_QC.csv",
    OUT / "Wu_binocular_pursuit_variability_participant.csv",
    OUT / "Wu_binocular_pursuit_variability_results.csv",
    OUT / "Wu_binocular_pursuit_variability_LOO.csv",
    OUT / "Wu_pursuit_variability_adjusted_for_median_gain.csv",
    OUT / "Wu_pursuit_variability_adjusted_for_mean_gain.csv",
    OUT / "Wu_pipeline_validation_participants.csv",
    OUT / "Wu_pipeline_validation_metrics.csv",
    OUT / "Wu_cleaning_sensitivity_summary.csv",
    OUT / "Wu_cleaning_sensitivity_participants.csv",
    OUT / "Wu_cleaning_sensitivity_trials.csv",
    OUT / "Wu_trial_count_uncertainty_participants.csv",
    OUT / "Wu_trial_count_sensitivity.csv",
    OUT / "Wu_trial_resampling_group_difference.csv",
    OUT / "Wu_trial_dynamics_by_trial.csv",
    OUT / "Wu_trial_dynamics_participant_slopes.csv",
    OUT / "Wu_trial_dynamics_clustered_OLS.csv",
]

print("\nExpected analysis outputs:")
for path in expected_outputs:
    status = "OK" if path.exists() else "MISSING"
    print(f"{status:7s} {path}")